# 08 — Variance swap pricing

Three routes to the fair variance strike:
1. Heston closed form (CIR mean integral).
2. Rough Bergomi closed form (just `int xi0(t) dt / T`).
3. Model-free static replication via a strip of OTM options.

All three should agree within wing-truncation error (~30 bps).

## Context

A variance swap pays $(\mathrm{RV}(0, T) - K_{\mathrm{var}})$ at maturity. The fair strike $K_{\mathrm{var}}$ has closed forms under both models:

- **Heston:** $K_{\mathrm{var}} = \theta + (v_0 - \theta)(1 - e^{-\kappa T}) / (\kappa T)$.
- **rBergomi:** $K_{\mathrm{var}} = \tfrac{1}{T} \int_0^T \xi_0(s)\, ds$ — directly in terms of the forward variance curve. Elegant, and one of rBergomi's selling points.

We cross-check both against the Demeterfi–Derman–Kamal–Zou (1999) model-free static replication,
$$K_{\mathrm{var}} = \frac{2 e^{rT}}{T}\Big[\int_0^F P(K)/K^2\,dK + \int_F^\infty C(K)/K^2\,dK\Big],$$
evaluated on the calibrated models' vanilla prices. The $e^{rT}$ factor converts present-value option prices back to forward payoffs — omitting it biases the strike low by exactly $e^{rT}$. Centering the strip on $K^\* = F$ cancels the deterministic log-contract terms.

In [ ]:
import numpy as np
import pandas as pd

from volengine.models.heston import HestonParameters, heston_vanilla_price
from volengine.models.rbergomi import RBergomiParameters
from volengine.products import (
    variance_swap_rate_heston, variance_swap_rate_rbergomi,
    variance_swap_rate_replication,
)

In [ ]:
h = HestonParameters(kappa=2.0, theta=0.04, xi=0.4, rho=-0.7, v0=0.04)
rb = RBergomiParameters(H=0.1, eta=1.9, rho=-0.9, xi0=0.04)
S0, r, q = 100.0, 0.03, 0.01

rows = []
for T in [0.25, 0.5, 1.0, 2.0]:
    F = S0 * np.exp((r - q) * T)
    K_grid = np.linspace(0.3 * F, 2.0 * F, 400)
    K_h_closed = variance_swap_rate_heston(h, T)
    K_h_repl = variance_swap_rate_replication(
        lambda K, T_=T: heston_vanilla_price(K, T_, S0, r, q, h, flag='call'),
        lambda K, T_=T: heston_vanilla_price(K, T_, S0, r, q, h, flag='put'),
        S0=S0, F=F, T=T, K_grid=K_grid, r=r,
    )
    K_rb_closed = variance_swap_rate_rbergomi(rb, T)
    rows.append({
        'T': T,
        'Heston (closed)': np.sqrt(K_h_closed),
        'Heston (replication)': np.sqrt(K_h_repl),
        'rBergomi (closed)': np.sqrt(K_rb_closed),
    })
pd.DataFrame(rows).round(4)

## Three-way agreement table

Closed form, model MC (under each model's calibrated parameters), and static replication should all agree within 30 bps. Discrepancies localize either to MC noise, wing truncation in the replication, or model misspecification.

## Term structure

**Figure.** Fair variance strike $K_{\mathrm{var}}(T)$ under each model as a function of $T$. Both models flatten toward the long-run variance $\theta$ (Heston) / curve-implied level (rBergomi); the short end is sensitive to $v_0$ / $\xi_0(0)$.